In [ ]:
# ============================================================
# LADDER — AML With vs Without Context, 8 panels (2×4)
# p1: Embedding win counts (LADDER split by confidence)
# p2: Cosine raincloud
# p3: ROUGE win counts (LADDER split by confidence)
# p4: ROUGE F1 raincloud
# ============================================================
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggdist)
library(scales)

# === FILE PATHS ===
aml_with <- list(
  emb   = "Intermediate Files/BREASTWITHCONTEXT_Semantic_results_general_only_withConfidence.csv",

  rouge = "Intermediate Files/Breastrouge__results/BreastCancerWithContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv",

  label = "Breast — With Context"
)


aml_without <- list(
  emb   = "Intermediate Files/BREASTWITHOUTCONTEXT_Semantic_results_general_only_withConfidence.csv",

  rouge = "Intermediate Files/Breastrouge__results/BreastCancerWithoutContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv",

  label = "Breast — Without Context"
)


#keep_models <- c("BioLORD-2023", "BioSentVec", "BiomedBERT", "MedCPT", "PubMedBERT")
keep_models <- c("BioLORD-2023", "MedCPT")


conf_breaks <- c(0, 0.45, 0.77, 0.97)
conf_labels <- c("LADDER — Low", "LADDER — Medium", "LADDER — High")

method_pal <- c(
  "LADDER — High"   = "#a50026", 
  "LADDER — Medium" = "#ca0020",
  "LADDER — Low"    = "#f4a582",
  "Hu et al"        = "#0571b0",
  "GeneAgent"       = "#4dac26"
)

all_levels <- c("LADDER — High", "LADDER — Medium", "LADDER — Low",
                "Hu et al", "GeneAgent")

theme_nature <- function(base_size = 10) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
  theme(
    panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
    panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
    panel.grid.minor   = element_blank(),
    axis.line          = element_blank(),
    axis.ticks         = element_line(colour = "black", linewidth = 0.45),
    axis.ticks.length  = unit(3, "pt"),
    axis.title         = element_text(face = "bold", size = base_size),
    axis.text          = element_text(colour = "black", size = base_size - 1),
    axis.text.x        = element_text(colour = "black", size = base_size - 1,
                                      angle = 0, hjust = 0.5),
    legend.title       = element_text(face = "bold", size = base_size - 1),
    legend.text        = element_text(size = base_size - 2),
    legend.key         = element_blank(),
    legend.background  = element_blank(),
    plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
    plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
    strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
    strip.text         = element_text(face = "bold", size = base_size - 1),
    plot.margin        = margin(8, 12, 8, 8)
  )
}

# ============================================================
# FUNCTION: 4 panels for one condition
# ============================================================
make_panels <- function(emb_path, rouge_path, disease_label) {

  emb <- read.csv(emb_path, stringsAsFactors = FALSE) %>%
    filter(Model %in% keep_models) %>%
    mutate(
      Winner = recode(Winner, "Our" = "LADDER", "Hu" = "Hu et al"),
      Model  = factor(Model, levels = keep_models)
    )

  wins_emb_ladder <- emb %>%
    filter(Winner == "LADDER") %>%
    mutate(
      Method = as.character(
        cut(Final_Confidence, breaks = conf_breaks, labels = conf_labels,
            include.lowest = TRUE, right = FALSE)
      ),
      Method = factor(Method, levels = all_levels)
    ) %>%
    count(Model, Method)

  wins_emb_others <- emb %>%
    filter(Winner != "LADDER") %>%
    mutate(Method = factor(Winner, levels = all_levels)) %>%
    count(Model, Method)


  bw      <- 0.115
  offsets <- c("LADDER" = -0.25, "Hu et al" = 0, "GeneAgent" = 0.25)
  model_idx <- setNames(seq_along(keep_models), keep_models)

  ladder_stack <- wins_emb_ladder %>%
    group_by(Model) %>%
    arrange(match(Method, conf_labels)) %>%
    mutate(
      ymax = cumsum(n),
      ymin = ymax - n,
      xmid = model_idx[as.character(Model)] + offsets["LADDER"],
      xmin = xmid - bw,
      xmax = xmid + bw
    ) %>% ungroup()

  ladder_totals <- ladder_stack %>%
    group_by(Model) %>%
    summarise(xmid = first(xmid), total = max(ymax), .groups = "drop")

  other_stack <- wins_emb_others %>%
    mutate(
      ymin = 0, ymax = n,
      xmid = model_idx[as.character(Model)] + offsets[as.character(Method)],
      xmin = xmid - bw, xmax = xmid + bw
    )

  ladder_seg_labels <- ladder_stack %>%
    mutate(ymid = (ymin + ymax) / 2,
           seg_n = n) %>%
    filter(seg_n > 0)

  p1 <- ggplot() +
    geom_rect(data = ladder_stack,
              aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = Method),
              colour = "white", linewidth = 0.4) +
    geom_rect(data = other_stack,
              aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = Method),
              colour = "white", linewidth = 0.4) +
    # Total on top of LADDER stack
    geom_text(data = ladder_totals,
              aes(x = xmid, y = total, label = total),
              vjust = -0.4, size = 2.5, fontface = "bold", colour = "black") +
    # Per-segment count inside each LADDER segment
    geom_text(data = ladder_seg_labels,
              aes(x = xmid, y = ymid, label = seg_n),
              size = 2.2, fontface = "bold", colour = "white") +
    # Labels on top of Hu et al / GeneAgent bars
    geom_text(data = other_stack,
              aes(x = xmid, y = ymax, label = n),
              vjust = -0.4, size = 2.5, fontface = "bold", colour = "black") +
    scale_fill_manual(values = method_pal, drop = FALSE) +
    scale_x_continuous(breaks = seq_along(keep_models), labels = keep_models) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15)),
                       breaks = pretty_breaks(4)) +
    labs(title    = disease_label,
         subtitle = "Embedding win counts",
         x = "Embedding Model", y = "Number of Wins",
         fill = "Method") +
    theme_nature() +
    theme(panel.grid.major.x = element_blank())

  sim_long <- emb %>%
    select(Model, Geneset,
           LADDER     = LADDER_Similarity,
           `Hu et al` = Hu_Similarity,
           GeneAgent  = GeneAgent_Similarity) %>%
    pivot_longer(c(LADDER, `Hu et al`, GeneAgent),
                 names_to = "Method", values_to = "Similarity") %>%
    mutate(Method = factor(Method, levels = c("LADDER", "Hu et al", "GeneAgent")),
           Model  = factor(Model,  levels = keep_models))

  raincloud_pal <- c("LADDER" = "#ca0020", "Hu et al" = "#0571b0", "GeneAgent" = "#4dac26")

  p2 <- ggplot(sim_long, aes(x = Model, y = Similarity,
                              fill = Method, colour = Method)) +
    stat_halfeye(adjust = 0.8, width = 0.4, .width = 0,
                 justification = -0.2, point_colour = NA, alpha = 0.72,
                 position = position_dodge(width = 0.7)) +
    geom_boxplot(outlier.shape = NA, width = 0.16, linewidth = 0.45,
                 colour = "black", alpha = 0.5,
                 position = position_dodge(width = 0.7)) +
    stat_dots(side = "left", dotsize = 0.55, binwidth = 0.013,
              alpha = 0.3, position = position_dodge(width = 0.7)) +
    scale_fill_manual(values = raincloud_pal, drop = FALSE) +
    scale_colour_manual(values = raincloud_pal, drop = FALSE) +
    scale_y_continuous(breaks = seq(0, 1, 0.2), limits = c(NA, 1.05)) +
    labs(title    = disease_label,
         subtitle = "Cosine similarity distribution",
         x = "Embedding Model", y = "Cosine Similarity") +
    theme_nature() +
    guides(fill = "none", colour = "none")

  rouge_raw <- read.csv(rouge_path, stringsAsFactors = FALSE)

  rouge_long <- rouge_raw %>%
    select(Geneset,
           LADDER_r1 = Our_rouge1_f,  Hu_r1 = Hu_rouge1_f,  GA_r1 = GeneAgent_rouge1_f,
           LADDER_r2 = Our_rouge2_f,  Hu_r2 = Hu_rouge2_f,  GA_r2 = GeneAgent_rouge2_f,
           LADDER_rL = Our_rougeL_f,  Hu_rL = Hu_rougeL_f,  GA_rL = GeneAgent_rougeL_f) %>%
    pivot_longer(-Geneset, names_to = c("Method", "ROUGE"), names_sep = "_",
                 values_to = "F1") %>%
    mutate(
      Method = recode(Method, "LADDER" = "LADDER", "Hu" = "Hu et al", "GA" = "GeneAgent"),
      Method = factor(Method, levels = c("LADDER", "Hu et al", "GeneAgent")),
      ROUGE  = recode(ROUGE, "r1" = "ROUGE-1", "r2" = "ROUGE-2", "rL" = "ROUGE-L"),
      ROUGE  = factor(ROUGE, levels = c("ROUGE-1", "ROUGE-2", "ROUGE-L"))
    )

  rouge_wins <- rouge_raw %>%
    select(Geneset, Final_Confidence,
           rouge1 = Best_method_by_rouge1_F1,
           rouge2 = Best_method_by_rouge2_F1,
           rougeL = Best_method_by_rougeL_F1) %>%
    pivot_longer(-c(Geneset, Final_Confidence),
                 names_to = "ROUGE", values_to = "Winner") %>%
    mutate(
      Winner = recode(Winner, "Our" = "LADDER", "Hu" = "Hu et al"),
      Method = case_when(
        Winner == "LADDER" ~ as.character(
          cut(Final_Confidence,
              breaks = conf_breaks,
              labels = conf_labels,
              include.lowest = TRUE, right = FALSE)
        ),
        TRUE ~ Winner
      ),
      Method = factor(Method, levels = all_levels),
      ROUGE  = recode(ROUGE,
                      "rouge1" = "ROUGE-1",
                      "rouge2" = "ROUGE-2",
                      "rougeL" = "ROUGE-L"),
      ROUGE  = factor(ROUGE, levels = c("ROUGE-1", "ROUGE-2", "ROUGE-L"))
    ) %>%
    count(ROUGE, Method)

  rouge_levels <- c("ROUGE-1", "ROUGE-2", "ROUGE-L")
  rouge_idx    <- setNames(seq_along(rouge_levels), rouge_levels)
  r_offsets    <- c("LADDER" = -0.25, "Hu et al" = 0, "GeneAgent" = 0.25)

  rouge_wins_ladder <- rouge_wins %>% filter(Method %in% conf_labels)
  rouge_wins_others <- rouge_wins %>% filter(!Method %in% conf_labels)

  r_ladder_stack <- rouge_wins_ladder %>%
    group_by(ROUGE) %>%
    arrange(match(Method, conf_labels)) %>% 
    mutate(
      ymax = cumsum(n), ymin = ymax - n,
      xmid = rouge_idx[as.character(ROUGE)] + r_offsets["LADDER"],
      xmin = xmid - bw, xmax = xmid + bw
    ) %>% ungroup()

  r_ladder_totals <- r_ladder_stack %>%
    group_by(ROUGE) %>%
    summarise(xmid = first(xmid), total = max(ymax), .groups = "drop")

  r_other_stack <- rouge_wins_others %>%
    mutate(
      ymin = 0, ymax = n,
      xmid = rouge_idx[as.character(ROUGE)] + r_offsets[as.character(Method)],
      xmin = xmid - bw, xmax = xmid + bw
    )

  r_ladder_seg_labels <- r_ladder_stack %>%
    mutate(ymid = (ymin + ymax) / 2,
           seg_n = n) %>%
    filter(seg_n > 0)

  p3 <- ggplot() +
    geom_rect(data = r_ladder_stack,
              aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = Method),
              colour = "white", linewidth = 0.4) +
    geom_rect(data = r_other_stack,
              aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = Method),
              colour = "white", linewidth = 0.4) +
    geom_text(data = r_ladder_totals,
              aes(x = xmid, y = total, label = total),
              vjust = -0.4, size = 2.5, fontface = "bold", colour = "black") +
    geom_text(data = r_ladder_seg_labels,
              aes(x = xmid, y = ymid, label = seg_n),
              size = 2.2, fontface = "bold", colour = "white") +
    geom_text(data = r_other_stack,
              aes(x = xmid, y = ymax, label = n),
              vjust = -0.4, size = 2.5, fontface = "bold", colour = "black") +
    scale_fill_manual(values = method_pal, drop = FALSE) +
    scale_x_continuous(breaks = seq_along(rouge_levels), labels = rouge_levels) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15)),
                       breaks = pretty_breaks(4)) +
    labs(title    = disease_label,
         subtitle = "ROUGE win counts",
         x = "ROUGE Metric", y = "Number of Wins",
         fill = "Method") +
    theme_nature() +
    theme(panel.grid.major.x = element_blank())

  p4 <- ggplot(rouge_long, aes(x = Method, y = F1,
                                fill = Method, colour = Method)) +
    stat_halfeye(adjust = 0.8, width = 0.4, .width = 0,
                 justification = -0.2, point_colour = NA, alpha = 0.72) +
    geom_boxplot(outlier.shape = NA, width = 0.16, linewidth = 0.45,
                 colour = "black", alpha = 0.5) +
    stat_dots(side = "left", dotsize = 0.5, binwidth = 0.015, alpha = 0.3) +
    facet_wrap(~ ROUGE, nrow = 1) +
    scale_fill_manual(values = raincloud_pal, drop = FALSE) +
    scale_colour_manual(values = raincloud_pal, drop = FALSE) +
    scale_y_continuous(breaks = seq(0, 0.4, 0.1), limits = c(0, 0.55),
                   labels = number_format(accuracy = 0.1)) +
    labs(title    = disease_label,
         subtitle = "ROUGE F1 distribution",
         x = NULL, y = "F1 Score") +
    theme_nature() +
    guides(fill = "none", colour = "none")

  list(p1 = p1, p2 = p2, p3 = p3, p4 = p4)
}

# ============================================================
# GENERATE PANELS
# ============================================================
pw <- make_panels(aml_with$emb,    aml_with$rouge,    aml_with$label)
po <- make_panels(aml_without$emb, aml_without$rouge, aml_without$label)

# ============================================================
# ASSEMBLE: 2 rows × 4 cols
# Row 1 = With Context, Row 2 = Without Context
# ============================================================
fig <- (po$p1 | po$p2 | po$p3 | po$p4) /
       (pw$p1 | pw$p2 | pw$p3 | pw$p4) +
  plot_annotation(
    tag_levels = "a",
    theme = theme(plot.tag = element_text(face = "bold", size = 12))
  ) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom")

ggsave("Fig_LADDER_BREAST_WithVsWithoutBiolordandMedCPT.pdf", fig, width = 28, height = 11, dpi = 300)
ggsave("Fig_LADDER_BREAST_WithVsWithoutBiolordandMedCPT.png", fig, width = 28, height = 11, dpi = 300)

cat("✓ Saved Fig_LADDER_BREAST_WithVsWithoutBiolordandMedCPT.pdf/.png\n")